# Kaggle Train: T5 Internal MVPruner-Style Ablations

Notebook nay dung branch `nguyet-t5-internal-mvpruner` va chay mot trong ba ablations A1/A2/A3. Checkpoint zips duoc ghi vao `/kaggle/working/checkpoint_zips` de co the tai xuong tu Kaggle output panel. Watcher se tu giu toi da 2 zip checkpoint moi nhat.

In [ ]:
GITHUB_REPO_URL = "https://github.com/huyvanzzz/Group_HND_car.git"
GITHUB_BRANCH = "nguyet-t5-internal-mvpruner"

# Chon: "a1_global_topk", "a2_per_view_uniform", hoac "a3_ira"
ABLATION = "a1_global_topk"

RUN_PROGRESS_MONITOR = False
EVAL_WITH_METEOR = False
RUN_ALIGN_STAGE = True
RUN_FINETUNE_STAGE = True
RUN_EVALUATE = True
RUN_BENCHMARK = True

RESUME_ALIGN_CHECKPOINT = ""      # vi du: /kaggle/input/my-run/align_latest.pt
RESUME_FINETUNE_CHECKPOINT = ""   # vi du: /kaggle/input/my-run/finetune_latest.pt

ABLATION_CONFIGS = {
    "a1_global_topk": "configs/repvit_t5_efficient_mini_internal_pruning_a1_global_topk.yaml",
    "a2_per_view_uniform": "configs/repvit_t5_efficient_mini_internal_pruning_a2_per_view_uniform.yaml",
    "a3_ira": "configs/repvit_t5_efficient_mini_internal_pruning_a3_ira.yaml",
}
ABLATION_PROFILES = {
    "a1_global_topk": "outputs/repvit_t5_efficient_mini_internal_pruning_a1_global_topk",
    "a2_per_view_uniform": "outputs/repvit_t5_efficient_mini_internal_pruning_a2_per_view_uniform",
    "a3_ira": "outputs/repvit_t5_efficient_mini_internal_pruning_a3_ira",
}

if ABLATION not in ABLATION_CONFIGS:
    raise ValueError(f"Unknown ABLATION: {ABLATION}")

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["ABLATION"] = ABLATION
os.environ["T5_PRUNING_CONFIG"] = ABLATION_CONFIGS[ABLATION]
os.environ["T5_PRUNING_PROFILE"] = ABLATION_PROFILES[ABLATION]
os.environ["RUN_PROGRESS_MONITOR"] = "1" if RUN_PROGRESS_MONITOR else "0"
os.environ["EVAL_WITH_METEOR"] = "1" if EVAL_WITH_METEOR else "0"
os.environ["RUN_ALIGN_STAGE"] = "1" if RUN_ALIGN_STAGE else "0"
os.environ["RUN_FINETUNE_STAGE"] = "1" if RUN_FINETUNE_STAGE else "0"
os.environ["RUN_EVALUATE"] = "1" if RUN_EVALUATE else "0"
os.environ["RUN_BENCHMARK"] = "1" if RUN_BENCHMARK else "0"
os.environ["RESUME_ALIGN_CHECKPOINT"] = RESUME_ALIGN_CHECKPOINT
os.environ["RESUME_FINETUNE_CHECKPOINT"] = RESUME_FINETUNE_CHECKPOINT
os.environ["CHECKPOINT_ZIP_DIR"] = "/kaggle/working/checkpoint_zips"
os.environ["MAX_CHECKPOINT_ZIPS"] = "2"

print({
    "repo": GITHUB_REPO_URL,
    "branch": GITHUB_BRANCH,
    "ablation": ABLATION,
    "config": os.environ["T5_PRUNING_CONFIG"],
    "profile": os.environ["T5_PRUNING_PROFILE"],
    "checkpoint_zip_dir": os.environ["CHECKPOINT_ZIP_DIR"],
    "max_checkpoint_zips": os.environ["MAX_CHECKPOINT_ZIPS"],
})

## 1. Clone repo va cai package

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only
python -m pip install -e ".[dev]"

## 2. Token va GPU check

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print({"hf_token_present": bool(os.environ.get("HF_TOKEN"))})

In [ ]:
import torch
count = torch.cuda.device_count()
print({"cuda": torch.cuda.is_available(), "gpu_count": count})
for idx in range(count):
    print(idx, torch.cuda.get_device_name(idx))
if count < 2:
    raise RuntimeError("Full train notebook expects Kaggle GPU T4 x2.")

In [ ]:
%%bash
set -e
nvidia-smi
ps -ef | grep -E "accelerate|efficient_vlm_ad|python" | grep -v grep || true

## 3. Sanity check config

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
python -m efficient_vlm_ad inspect-data --config "$CONFIG"
python - <<'PY'
import os
from pathlib import Path
import yaml

config_path = Path(os.environ["T5_PRUNING_CONFIG"])
cfg = yaml.safe_load(config_path.read_text())
summary = {
    "config": str(config_path),
    "output_dir": cfg["project"]["output_dir"],
    "profile": cfg["model"]["profile"],
    "architecture": cfg["model"].get("architecture"),
    "selection_policy": cfg["model"]["pruning"].get("selection_policy"),
    "layer_policy": cfg["model"]["pruning"].get("layer_policy"),
    "keep_ratio": cfg["model"]["pruning"].get("keep_ratio"),
    "min_keep_per_view": cfg["model"]["pruning"].get("min_keep_per_view"),
    "batch_size_per_gpu": cfg["training"]["batch_size"],
    "gradient_accumulation_steps": cfg["training"]["gradient_accumulation_steps"],
    "effective_batch_2gpu": cfg["training"]["batch_size"] * cfg["training"]["gradient_accumulation_steps"] * 2,
}
print(summary)
assert cfg["model"].get("architecture") == "t5_internal_pruning"
assert cfg["model"]["pruning"].get("layer_policy") == "middle"
assert cfg["model"]["pruning"].get("keep_ratio") == 0.5
PY

## 4. Prepare data va cache frozen visual features

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset full
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset full

## 5. Start checkpoint zip watcher

Watcher nay nen duoc chay truoc train. Moi khi `align_latest.pt` hoac `finetune_latest.pt` doi sau epoch, watcher tao zip moi trong `/kaggle/working/checkpoint_zips` va xoa zip cu de luon giu toi da 2 zip.

In [ ]:
import os
import subprocess
import sys
import textwrap
from pathlib import Path

watcher_script = Path("/kaggle/working/checkpoint_zip_watcher.py")
watcher_script.write_text(textwrap.dedent(r'''
    import os
    import time
    import zipfile
    from datetime import datetime, timezone
    from pathlib import Path

    repo = Path('/kaggle/working/Efficient_VLM_For_Autonomous_Driving')
    profile = repo / os.environ['T5_PRUNING_PROFILE']
    checkpoints = profile / 'checkpoints'
    zip_dir = Path(os.environ.get('CHECKPOINT_ZIP_DIR', '/kaggle/working/checkpoint_zips'))
    max_zips = int(os.environ.get('MAX_CHECKPOINT_ZIPS', '2'))
    ablation = os.environ.get('ABLATION', 'ablation')
    zip_dir.mkdir(parents=True, exist_ok=True)
    seen = {}

    def prune_old_zips():
        zips = sorted(zip_dir.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
        for old in zips[max_zips:]:
            old.unlink(missing_ok=True)

    def add_if_exists(zf, path, arcname=None):
        if path.exists():
            zf.write(path, arcname or str(path.relative_to(repo)))

    def make_zip(checkpoint_path):
        stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        name = f'{ablation}_{checkpoint_path.stem}_{stamp}.zip'
        target = zip_dir / name
        tmp = target.with_suffix('.zip.tmp')
        with zipfile.ZipFile(tmp, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            add_if_exists(zf, checkpoint_path)
            for extra in ('align_best.pt', 'finetune_best.pt', 'best_model.pt'):
                add_if_exists(zf, checkpoints / extra)
            add_if_exists(zf, profile / 'debug' / 'train_progress.jsonl')
            add_if_exists(zf, profile / 'debug' / 'debug_events.jsonl')
            add_if_exists(zf, repo / os.environ['T5_PRUNING_CONFIG'])
        tmp.replace(target)
        prune_old_zips()
        print(f'[checkpoint-zip] wrote {target}', flush=True)

    prune_old_zips()
    print(f'[checkpoint-zip] watching {checkpoints}', flush=True)
    while True:
        for checkpoint_path in (checkpoints / 'align_latest.pt', checkpoints / 'finetune_latest.pt'):
            if not checkpoint_path.exists():
                continue
            stat = checkpoint_path.stat()
            signature = (stat.st_mtime_ns, stat.st_size)
            if seen.get(str(checkpoint_path)) != signature:
                time.sleep(5)
                new_stat = checkpoint_path.stat()
                stable_signature = (new_stat.st_mtime_ns, new_stat.st_size)
                if stable_signature == signature:
                    seen[str(checkpoint_path)] = signature
                    make_zip(checkpoint_path)
        time.sleep(20)
'''), encoding='utf-8')

log_path = Path('/kaggle/working/checkpoint_zip_watcher.log')
log_file = log_path.open('a', encoding='utf-8')
checkpoint_zip_watcher = subprocess.Popen(
    [sys.executable, str(watcher_script)],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
print({"watcher_pid": checkpoint_zip_watcher.pid, "log": str(log_path), "zip_dir": os.environ['CHECKPOINT_ZIP_DIR']})

## 6. Focused tests va 1-step train probe

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
PROFILE="$T5_PRUNING_PROFILE"

python -m pytest tests/test_config.py tests/test_modeling.py -q
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1 --debug-numerics

if [ -n "${RESUME_ALIGN_CHECKPOINT:-}" ] || [ -n "${RESUME_FINETUNE_CHECKPOINT:-}" ] || [ -f "$PROFILE/checkpoints/align_latest.pt" ] || [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  echo "Skipping 1-step 2GPU probe because a resume checkpoint is configured or already exists."
else
  PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 1 --debug --debug-samples 1 --debug-numerics
  rm -f "$PROFILE/checkpoints/align_latest.pt" "$PROFILE/checkpoints/align_best.pt"
fi

## 7. Train stage align

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
PROFILE="$T5_PRUNING_PROFILE"

if [ "${RUN_ALIGN_STAGE:-1}" != "1" ]; then
  echo "Skipping align stage because RUN_ALIGN_STAGE=0."
  exit 0
fi

RESUME_ARGS=()
if [ -n "${RESUME_ALIGN_CHECKPOINT:-}" ]; then
  if [ ! -f "$RESUME_ALIGN_CHECKPOINT" ]; then
    echo "Missing RESUME_ALIGN_CHECKPOINT: $RESUME_ALIGN_CHECKPOINT"
    exit 1
  fi
  RESUME_ARGS=(--resume "$RESUME_ALIGN_CHECKPOINT")
elif [ -f "$PROFILE/checkpoints/align_latest.pt" ]; then
  RESUME_ARGS=(--resume "$PROFILE/checkpoints/align_latest.pt")
fi

PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align "${RESUME_ARGS[@]}" --debug --debug-samples 1 --debug-numerics

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$T5_PRUNING_PROFILE"
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true
echo "Checkpoint zips:"
ls -lh /kaggle/working/checkpoint_zips || true
tail -n 20 /kaggle/working/checkpoint_zip_watcher.log || true

## 8. Train stage finetune

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
PROFILE="$T5_PRUNING_PROFILE"

if [ "${RUN_FINETUNE_STAGE:-1}" != "1" ]; then
  echo "Skipping finetune stage because RUN_FINETUNE_STAGE=0."
  exit 0
fi

if [ -n "${RESUME_FINETUNE_CHECKPOINT:-}" ]; then
  RESUME="$RESUME_FINETUNE_CHECKPOINT"
elif [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  RESUME="$PROFILE/checkpoints/finetune_latest.pt"
else
  RESUME="$PROFILE/checkpoints/align_best.pt"
fi

if [ ! -f "$RESUME" ]; then
  echo "Missing resume checkpoint: $RESUME"
  echo "Run align first, set RESUME_FINETUNE_CHECKPOINT, or upload/copy the expected checkpoint."
  exit 1
fi

PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$RESUME" --debug --debug-samples 1 --debug-numerics

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$T5_PRUNING_PROFILE"
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
python -m efficient_vlm_ad monitor-progress --output-dir "$PROFILE" --last 5 || true
echo "Checkpoint zips:"
ls -lh /kaggle/working/checkpoint_zips || true
tail -n 20 /kaggle/working/checkpoint_zip_watcher.log || true

## 9. Evaluate va benchmark

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
CONFIG="$T5_PRUNING_CONFIG"
PROFILE="$T5_PRUNING_PROFILE"
CHECKPOINT="$PROFILE/checkpoints/best_model.pt"
if [ ! -f "$CHECKPOINT" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
fi
if [ ! -f "$CHECKPOINT" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"
fi
if [ ! -f "$CHECKPOINT" ]; then
  echo "Missing evaluation checkpoint: $CHECKPOINT"
  exit 1
fi
METEOR_FLAG=""
if [ "${EVAL_WITH_METEOR:-0}" = "1" ]; then METEOR_FLAG="--with-meteor"; fi
if [ "${RUN_EVALUATE:-1}" = "1" ]; then
  python -m efficient_vlm_ad evaluate --config "$CONFIG" --checkpoint "$CHECKPOINT" --batch-size "${EVAL_BATCH_SIZE:-16}" --progress-log-every-samples "${EVAL_PROGRESS_LOG_EVERY:-256}" $METEOR_FLAG
fi
if [ "${RUN_BENCHMARK:-1}" = "1" ]; then
  python -m efficient_vlm_ad benchmark --config "$CONFIG" --checkpoint "$CHECKPOINT" --progress-log-every-samples "${BENCHMARK_PROGRESS_LOG_EVERY:-20}"
fi

## 10. Artifacts va dung watcher

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
PROFILE="$T5_PRUNING_PROFILE"
ls -lh "$PROFILE/checkpoints" || true
ls -lh /kaggle/working/checkpoint_zips || true
cat "$PROFILE/metrics.json" 2>/dev/null || true
cat "$PROFILE/benchmark.json" 2>/dev/null || true

In [ ]:
try:
    checkpoint_zip_watcher.terminate()
    checkpoint_zip_watcher.wait(timeout=10)
    print({"watcher_stopped": True, "returncode": checkpoint_zip_watcher.returncode})
except NameError:
    print({"watcher_stopped": False, "reason": "watcher process variable not found"})
except Exception as exc:
    print({"watcher_stopped": False, "error": str(exc)})